## CHARGEMENT DES PACKAGES

In [ ]:
from Modules.visual_analyser import VisualAnalyzer
from Modules.data_preprocessing import DataPreprocessing
from Modules.image_analyzer import ImageAnalyzer
import pandas as pd
import os
import random
import numpy as np
from Modules.image_hasing import DuplicateHashEvaluator
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
# recupération des données
dp = DataPreprocessing()
source_path = ""
df_path = os.path
df = dp.read_data(os.path.join(source_path, "post_rehydrated.pickle"), format_="pickle")
dp.parse_dates()
# filtrage des données (On ne garde que les commentaires et quotes pas les originaux)
df = df[df["join_post_post_type"] != "original"].copy()

## TRAITEMENT DES DONNEES

In [ ]:
## Gestion des images dupliquées, NA etc ..
cols = ["image_name", "source_image_name"]

for col in cols:
    df.loc[:, col] = df[col].apply(
        lambda x: x if isinstance(x, list) else ([x] if pd.notna(x) else [])
    )


## sauvegarde dans fichiers séparés des fichiers séparements
df_pf_account_id_exploded = df.explode('image_name')
df_pf_account_id_exploded.to_csv('data/df_pf_caccount_id_exploded.csv')

df_source_pf_account_id_exploded = df.explode('source_image_name')
df_source_pf_account_id_exploded.to_csv('data/df_source_pf_caccount_id_exploded.csv')

## explode all
df_all_images = pd.concat([
    df[['image_name']].explode('image_name')
        .rename(columns={'image_name': 'image'}),
    df[['source_image_name']].explode('source_image_name')
        .rename(columns={'source_image_name': 'image'})
], ignore_index=True)

df_all_images = (
    df_all_images
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

df_all_images.to_csv("data/all_images.csv", index=False)

In [ ]:
len(df_all_images), len(df)

### GESTION DES IMAGES ET DETECTION DES IMAGES CASI-EXACTES

In [ ]:
## Gestion des images identiques ou quasi-exactes
img_analyzer = ImageAnalyzer(df=df)
source_path = "."
def image_loader_local(img):
    img_path = None
    images_dir = [
        os.path.join(source_path, os.path.join(d, d)) for d in ["img", "source_img"]
    ]
    for d in images_dir:
        candidate = os.path.join(source_path, d, d, img)
        if os.path.exists(candidate):
            img_path = candidate
            break

    if img_path is None:
        raise
    img = Image.open(img_path).convert("RGB")
    return img

## images exactes ou quasi-exactes
from PIL import Image

dup_hash_ev = DuplicateHashEvaluator(
    images_list=df_all_images['image'].unique().tolist(),
    labels_gt=df_all_images['image'].unique().tolist(),
    image_loader=image_loader_local,
    random_state=SEED,
)
images_clusters_dict = dup_hash_ev.find_images_cluster(method="phash", threshold=8)


In [ ]:
# sauvegarde dans un dataframe
rows = [(cluster_id, image) for cluster_id, images in images_clusters_dict.items() for image in images]

df_clusters_dup = pd.DataFrame(rows, columns=["cluster_id", "image_id"])

df_clusters_dup.to_csv('data/df_clusters_duplicated.csv')

In [ ]:
images_clusters_dict2 = dup_hash_ev.find_images_cluster(method="phash", threshold=8)

In [ ]:
images_clusters_dict2.equals(images_clusters_dict)

In [ ]:
images_clusters_dict2

### MERGING DES IMAGES

In [ ]:
len(images_clusters_dict[127])

In [ ]:
img_analyzer.display_specific_imgs(
    image_names=images_clusters_dict[127],
    source_path=source_path,
    ncols=7,
    from_internet=False,
    images_dir=['img', 'source_img']
)

In [ ]:
df_all_images.rename(columns={'image': 'image_id'}, inplace=True)
dup_hash_ev1 = DuplicateHashEvaluator(
    images_list=df_all_images['image_id'].unique().tolist(),
    labels_gt=df_all_images['image_id'].unique().tolist(),
    image_loader=image_loader_local,
    random_state=SEED,
)

dup_hash_ev2 = DuplicateHashEvaluator(
    images_list=df_all_images['image_id'].unique().tolist(),
    labels_gt=df_all_images['image_id'].unique().tolist(),
    image_loader=image_loader_local,
    random_state=SEED,
)

data_dup_images_1 = dup_hash_ev1.merge_dup_images(
    clusters_dict=images_clusters_dict,
    data=df_all_images.copy()
)

data_dup_images_2 = dup_hash_ev2.merge_dup_images(
    clusters_dict=images_clusters_dict,
    data=df_all_images.copy()
)

print(data_dup_images_1.equals(data_dup_images_2))
data_dup_images_1.to_csv('data/df_dup_images_merged.csv')

## GENERATION DES EMBEDDINGS CLIPS

In [ ]:
import torch
print(torch.cuda.is_available())
import clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

# Charger le modèle CLIP et sa fonction de preprocess
clip_model, preprocess = clip.load("ViT-B/32", device=device)

import torch
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn

def compute_clip_embeddings(
    images_list: list,
    image_loader,
    clip_model,
    preprocess,
    device="cuda",
    batch_size=32,
    use_half=False,
    seed=SEED
):
    """
    Calcule les embeddings CLIP pour une liste d'images.
    - Batch processing pour accélérer sur GPU
    - Option float16 pour réduire la mémoire
    - Barre de progression avec rich
    - Skip les images introuvables ou corrompues
    - Déterministe avec seed + cuDNN

    Args:
        images_list (list): liste des chemins/fichiers images
        image_loader (callable): fonction qui retourne PIL.Image
        clip_model: modèle CLIP chargé
        preprocess: fonction preprocess CLIP
        device (str): "cuda" ou "cpu"
        batch_size (int): taille du batch GPU
        use_half (bool): True pour float16
        seed (int): seed pour reproductibilité

    Returns:
        dict: {image_id: embedding_tensor (CPU)}
    """

    # === Seed et determinisme ===
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    if use_half:
        clip_model = clip_model.half()

    clip_model.to(device)
    clip_model.eval()

    image_embeddings = {}

    # === Progress bar Rich ===
    with Progress(
        SpinnerColumn(),
        "[progress.description]{task.description}",
        BarColumn(),
        TextColumn("[progress.percentage]{task.percentage:>3.0f}%"),
        TimeElapsedColumn(),
    ) as progress:

        task = progress.add_task("[green]Computing CLIP embeddings...", total=len(images_list))

        for i in range(0, len(images_list), batch_size):
            batch_ids = images_list[i:i+batch_size]
            batch_tensors = []
            valid_ids = []

            for img_id in batch_ids:
                if img_id in image_embeddings:
                    continue
                try:
                    img = image_loader(img_id)
                    img_tensor = preprocess(img)
                    if use_half:
                        img_tensor = img_tensor.half()
                    batch_tensors.append(img_tensor)
                    valid_ids.append(img_id)
                except FileNotFoundError:
                    print(f"[Warning] Image introuvable : {img_id}, skipping...")
                    continue
                except Exception as e:
                    print(f"[Error] Erreur embedding image {img_id}: {e}")
                    continue

            if not batch_tensors:
                progress.update(task, advance=len(batch_ids))
                continue

            batch_tensor = torch.stack(batch_tensors).to(device)

            with torch.no_grad():
                batch_emb = clip_model.encode_image(batch_tensor)

            for img_id, emb in zip(valid_ids, batch_emb):
                image_embeddings[img_id] = emb.cpu()

            progress.update(task, advance=len(batch_ids))

    return image_embeddings

In [ ]:
print(device)

In [ ]:
len(data_dup_images_1['unique_dup_img_name'].unique())

In [ ]:
images_list = data_dup_images_1['unique_dup_img_name'].unique()
embeddings = compute_clip_embeddings(
    images_list=images_list,
    image_loader=image_loader_local,  # fonction qui retourne PIL.Image
    clip_model=clip_model,
    preprocess=preprocess,
    device=device,
    batch_size=32,    # ajustable selon VRAM (32 safe pour 6GB)
    use_half=False,   # True si tu veux float16 (réduit la mémoire)
    seed=42           # pour reproductibilité
)
len(embeddings)

In [ ]:
## sauvegarder les embeddings
df_embeddings = pd.DataFrame(
    list(embeddings.items()),
    columns=['image_id', 'embedding']
)

# sauvegarder embeddings et ids séparément
image_ids = df_embeddings['image_id'].values
embeddings = np.stack(df_embeddings['embedding'].values)  # shape (8699, 512)
np.save('data/image_ids.npy', image_ids)
np.save('data/embeddings.npy', embeddings)

## CLUSTERING

In [ ]:
import numpy as np
import hdbscan
import pandas as pd
from sklearn.metrics import silhouette_score
from Modules.umap import UmapDimensionReducer


class HdbscanClusterer:
    """
    Clusterisation avec HDBSCAN + UMAP
    Version avec fit direct et score composite robuste.
    """

    def __init__(
        self,
        metric="euclidean",
        random_state=42,
    ):
        self.metric = metric
        self.random_state = random_state

        self.best_model = None
        self.best_labels = None
        self.best_params = None
        self.best_scores = None

    # -----------------------------------------------------
    # FIT DIRECT
    # -----------------------------------------------------
    def fit(
        self,
        data,
        n_neighbors: int,
        n_components: int,
        min_cluster_size: int,
        min_dist: float = 0.1,
        cluster_selection_method: str = "eom",
        metric_eval: str = "dbcv",
        min_samples_ratio: float = 0.5,
    ):
        """
        Fit UMAP + HDBSCAN sans Bayesian Optimization.
        """

        # ------------------------
        # UMAP
        # ------------------------
        reducer = UmapDimensionReducer(
            data=data,
            n_components=n_components,
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            random_state=self.random_state,
        )
        X_umap = reducer.fit_transform()

        # ------------------------
        # HDBSCAN
        # ------------------------
        min_samples = max(2, int(min_cluster_size * min_samples_ratio))

        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric=self.metric,
            cluster_selection_method=cluster_selection_method,
            gen_min_span_tree=True,
        )

        labels = clusterer.fit_predict(X_umap)

        # ------------------------
        # Evaluation
        # ------------------------
        if metric_eval.lower() == "dbcv":
            metric_val = clusterer.relative_validity_
        else:
            mask = labels != -1
            if len(set(labels[mask])) <= 1:
                metric_val = -1
            else:
                metric_val = silhouette_score(X_umap[mask], labels[mask])

        noise_ratio = np.mean(labels == -1)

        stability = (
            np.mean(clusterer.cluster_persistence_)
            if len(clusterer.cluster_persistence_) > 0
            else 0.0
        )

        # ------------------------
        # Score composite robuste
        # ------------------------
        composite_score = (
            0.5* metric_val
            + 0.5 * stability
            - 0.35 * noise_ratio
        )

        # ------------------------
        # Stockage
        # ------------------------
        self.best_model = clusterer
        self.best_labels = labels
        self.best_params = {
            "n_neighbors": n_neighbors,
            "min_dist": min_dist,
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
            "n_components": n_components,
            "cluster_selection_method": cluster_selection_method,
        }

        self.best_scores = {
            "metric_val": metric_val,
            "noise_ratio": noise_ratio,
            "stability": stability,
            "composite_score": composite_score,
        }

        return self

    # -----------------------------------------------------
    # Résumé
    # -----------------------------------------------------
    def summary(self):
        if self.best_labels is None:
            raise RuntimeError("Appeler fit() avant summary().")

        return {
            "best_params": self.best_params,
            "scores": self.best_scores,
            "n_clusters": len(set(self.best_labels))
            - (1 if -1 in self.best_labels else 0),
        }

    # -----------------------------------------------------
    # DataFrame résultats
    # -----------------------------------------------------
    def get_results(self, index=None):
        if self.best_labels is None:
            raise RuntimeError("Appeler fit() avant get_results().")

        if index is None:
            index = range(len(self.best_labels))

        return pd.DataFrame({"id": index, "cluster": self.best_labels})


In [ ]:
# recharger
image_ids = np.load('image_ids.npy', allow_pickle=True)
embeddings = np.load('embeddings.npy', allow_pickle= True)
emb_matrix = np.array(
    [emb for emb in embeddings]
)

img_ids = image_ids

#clusterer.relative_validity_
df_embeddings = pd.DataFrame(
    emb_matrix,
    columns=[f"dim_{i}" for i in range(emb_matrix.shape[1])],
    index=img_ids,
)

df_embeddings.index.name = "unique_dup_img_name"

### TESTING ....

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=df_embeddings,
    n_neighbors=5,
    n_components=15,
    min_cluster_size=100,  # juste assez pour créer des clusters stables
    min_dist=0.01,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=df_embeddings,
    n_neighbors=5,
    n_components=15,
    min_cluster_size=45,  # juste assez pour créer des clusters stables
    min_dist=0.01,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=df_embeddings,
    n_neighbors=5,
    n_components=20,
    min_cluster_size=40,  # juste assez pour créer des clusters stables
    min_dist=0.01,
    min_samples_ratio=0.2,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=df_embeddings,
    n_neighbors=5,
    n_components=20,
    min_cluster_size=40,  # juste assez pour créer des clusters stables
    min_dist=0.02,
    min_samples_ratio=0.2,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=df_embeddings,
    n_neighbors=5,
    n_components=20,
    min_cluster_size=45,  # juste assez pour créer des clusters stables
    min_dist=0.02,
    min_samples_ratio=0.2,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=df_embeddings,
    n_neighbors=5,
    n_components=25,
    min_cluster_size=45,  # juste assez pour créer des clusters stables
    min_dist=0.02,
    min_samples_ratio=0.2,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=df_embeddings,
    n_neighbors=5,
    n_components=25,
    min_cluster_size=35,  # juste assez pour créer des clusters stables
    min_dist=0.02,
    min_samples_ratio=0.2,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=df_embeddings,
    n_neighbors=10,
    n_components=25,
    min_cluster_size=35,  # juste assez pour créer des clusters stables
    min_dist=0.02,
    min_samples_ratio=0.2,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=df_embeddings,
    n_neighbors=10,
    n_components=15,
    min_cluster_size=40,  # juste assez pour créer des clusters stables
    min_dist=0.02,
    min_samples_ratio=0.2,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=df_embeddings,
    n_neighbors=5,
    n_components=25,
    min_cluster_size=50,  # juste assez pour créer des clusters stables
    min_dist=0.01,
    min_samples_ratio=0.2,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=df_embeddings,
    n_neighbors=5,
    n_components=25,
    min_cluster_size=50,  # juste assez pour créer des clusters stables
    min_dist=0.01,
    min_samples_ratio=0.1,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=df_embeddings,
    n_neighbors=5,
    n_components=25,
    min_cluster_size=50,  # juste assez pour créer des clusters stables
    min_dist=0.01,
    min_samples_ratio=0.3,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

## BAYESIAN OPTIMISATION

In [ ]:
import warnings
warnings.filterwarnings('ignore') # y'a juste un warning qui parle de redefinition du seed

In [ ]:
# hyperparameters space
## Bonne plages
min_cluster_size = [40, 50]
min_dist = [0.01, 0.05]
min_sample_ratio = [0.1, 0.3] # plus ou moins dense
n_components = [10, 25]
n_neighbors = [5, 10] # local car sujet pareil
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn
from skopt.utils import use_named_args

from skopt.space import Integer, Real
from skopt.utils import use_named_args
from Modules.bayesian_optimization import BayesianOptimization

space = [
    Integer(min(n_neighbors), max(n_neighbors), name="n_neighbors"),
    Integer(min(n_components), max(n_components), name="n_components"),
    Integer(min(min_cluster_size), max(min_cluster_size), name="min_cluster_size"),
    Real(min(min_sample_ratio), max(min_sample_ratio), name="min_samples_ratio"),
    Real(min(min_dist), max(min_dist), name="min_dist"),
]


progress = Progress(
    SpinnerColumn(),
    "[progress.description]{task.description}",
    BarColumn(),
    TextColumn("{task.completed}/{task.total}"),
    TimeElapsedColumn()
)

iteration = 0
total_calls = 55

task = progress.add_task(
    "[green]Running Bayesian Optimization...",
    total=total_calls
)

@use_named_args(space)
def objective(n_neighbors, n_components, min_cluster_size, min_samples_ratio, min_dist):
    global iteration
    iteration += 1
    progress.update(task, advance=1)

    # Fit HDBSCAN + UMAP
    clusterer = HdbscanClusterer(metric="euclidean")
    clusterer.fit(
        data=df_embeddings,
        n_neighbors=int(n_neighbors),
        n_components=int(n_components),
        min_cluster_size=int(min_cluster_size),
        min_dist=min_dist,
        min_samples_ratio=min_samples_ratio
    )

    # Récupération des métriques
    return -clusterer.best_scores["metric_val"]

# ------------------------------
# Lancement de l'optimisation
# ------------------------------
bopt_all_data = BayesianOptimization(
    objective_func=objective,
    space=space,
    n_calls=total_calls,
    n_initial_points=10,
    acq_func="EI",
    random_state=SEED,
)

with progress:
    bopt_all_data.run_optimization()

In [ ]:
best_params_bopt = {k: int(v) if isinstance(v, np.integer) else v for k, v in bopt_all_data.best_params.items()}
best_dbcv = bopt_all_data.best_score

best_params_clean = {
    k: int(v) if isinstance(v, (np.integer,))
    else float(v) if isinstance(v, (np.floating,))
    else v
    for k, v in best_params_bopt.items()
}
print(best_params_clean)

In [ ]:
final_clusterer = HdbscanClusterer(metric="euclidean")
final_clusterer.fit(
    data=df_embeddings,
    **best_params_clean
)
final_clusterer.summary()

In [ ]:
# recupération des résultats
df_hdbscan_clusters_level1 = final_clusterer.get_results()
df_embeddings.to_csv('data/df_embeddings.csv') # sauvegarde des embeddings
assert len(df_embeddings) == len(df_hdbscan_clusters_level1) # verification

# savegarde des resultats du premier clustering
df_hdbscan_clusters_level1 = df_hdbscan_clusters_level1.assign(
    unique_dup_img_name=df_embeddings.index
)
df_hdbscan_clusters_level1.to_csv('data/cluster_1_hdbscan.csv')

# sauvegarde des resultats du premier clustering sans les bruits
df_hdbscan_clusters_level_first = df_hdbscan_clusters_level1[df_hdbscan_clusters_level1['cluster'] != -1].copy()
df_hdbscan_clusters_level_first.to_csv('data/cluster_1_hdbscan_sans_bruit.csv')

# jointure des resultats
data_dup_images_1.rename(columns={'cluster': 'cluster_duplicated'}, inplace=True)
df_hdbcan_cluster_step_1_without_noises = df_hdbscan_clusters_level_first.\
    merge(
        right=data_dup_images_1,
        on="unique_dup_img_name",
        how="left"
    )
df_hdbcan_cluster_step_1_without_noises.rename(columns={'cluster': 'cluster_hdbscan'}, inplace=True)
df_hdbcan_cluster_step_1_without_noises.to_csv('data/df_hdbcan_cluster_step_1_without_noises.csv') # a considerer

# verif
print(len(df_hdbcan_cluster_step_1_without_noises['unique_dup_img_name'].unique()), len(data_dup_images_1['unique_dup_img_name'].unique()))
print(df_hdbcan_cluster_step_1_without_noises['image_id'].isna().sum(), len(df_hdbcan_cluster_step_1_without_noises['image_id'].unique()))

In [ ]:
df_hdbcan_cluster_step_1_without_noises.head()

In [ ]:
df_hdbcan_cluster_step_1_without_noises['cluster_hdbscan'].unique(), len(df_hdbcan_cluster_step_1_without_noises['cluster_hdbscan'].unique())

## Les Bruits

In [ ]:
noise_mask = final_clusterer.best_labels == -1
X_noise = df_embeddings[noise_mask]

#### TEST DES PLAGES

In [ ]:
clusterer = HdbscanClusterer()
clusterer.fit(
    data=X_noise,
    n_neighbors=8,
    n_components=25,
    min_cluster_size=25,  # juste assez pour créer des clusters stables
    min_dist=0.01,
    min_samples_ratio=0.2,
    cluster_selection_method="eom"
)

df_clusters = clusterer.get_results()
clusterer.summary()

In [ ]:
for i in df_clusters['cluster'].unique().tolist():
    print(i, len(df_clusters[df_clusters['cluster'] == i]))

In [ ]:
# sauvegarede des noises embeddings
X_noise.to_csv('data/df_embeddings_noise.csv')

In [ ]:
# hyperparameters space
## Bonne plages
min_cluster_size = [22, 25]
min_dist = [0.01, 0.02]
min_sample_ratio = [0.1, 0.2] # plus ou moins dense
n_components = [20, 25]
n_neighbors = [8, 9] # local car sujet pareil
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn
from skopt.utils import use_named_args

from skopt.space import Integer, Real
from skopt.utils import use_named_args
from Modules.bayesian_optimization import BayesianOptimization

space = [
    Integer(min(n_neighbors), max(n_neighbors), name="n_neighbors"),
    Integer(min(n_components), max(n_components), name="n_components"),
    Integer(min(min_cluster_size), max(min_cluster_size), name="min_cluster_size"),
    Real(min(min_sample_ratio), max(min_sample_ratio), name="min_samples_ratio"),
    Real(min(min_dist), max(min_dist), name="min_dist"),
]


progress = Progress(
    SpinnerColumn(),
    "[progress.description]{task.description}",
    BarColumn(),
    TextColumn("{task.completed}/{task.total}"),
    TimeElapsedColumn()
)

iteration = 0
total_calls = 55

task = progress.add_task(
    "[green]Running Bayesian Optimization...",
    total=total_calls
)
@use_named_args(space)
def objective(n_neighbors, n_components, min_cluster_size, min_samples_ratio, min_dist):
    global iteration
    iteration += 1
    progress.update(task, advance=1)

    # Fit HDBSCAN + UMAP
    clusterer = HdbscanClusterer(metric="euclidean")
    clusterer.fit(
        data=df_embeddings,
        n_neighbors=int(n_neighbors),
        n_components=int(n_components),
        min_cluster_size=int(min_cluster_size),
        min_dist=min_dist,
        min_samples_ratio=min_samples_ratio
    )

    # Récupération des métriques
    return -clusterer.best_scores["metric_val"]

# ------------------------------
# Lancement de l'optimisation
# ------------------------------
bopt_noise_data = BayesianOptimization(
    objective_func=objective,
    space=space,
    n_calls=total_calls,
    n_initial_points=10,
    acq_func="EI",
    random_state=SEED,
)

with progress:
    bopt_noise_data.run_optimization()

In [ ]:
best_noise_params_bopt = {k: int(v) if isinstance(v, np.integer) else v for k, v in bopt_noise_data.best_params.items()}
best_noise_dbcv = bopt_noise_data.best_score

best_noise_params_clean = {
    k: int(v) if isinstance(v, (np.integer,))
    else float(v) if isinstance(v, (np.floating,))
    else v
    for k, v in best_noise_params_bopt.items()
}
print(best_noise_params_clean)

In [ ]:
final_clusterer_noise = HdbscanClusterer(metric="euclidean")
final_clusterer_noise.fit(
    data=X_noise,
    **best_noise_params_clean
)
final_clusterer_noise.summary()

In [ ]:
df_hdbscan_clusters_noise = final_clusterer_noise.get_results()
df_hdbscan_clusters_noise.to_csv('data/cluster_noise_hdbscan.csv')

In [ ]:
# sauvegarde des resultats du premier clustering sans les bruits
df_hdbscan_clusters_noise = df_hdbscan_clusters_noise.assign(
    unique_dup_img_name=X_noise.index
)
df_hdbscan_clusters_level_second = df_hdbscan_clusters_noise[df_hdbscan_clusters_noise['cluster'] != -1].copy()
df_hdbscan_clusters_level_second['cluster'] = df_hdbscan_clusters_level_second['cluster'].\
    apply(lambda x: x + 1 + len(df_hdbcan_cluster_step_1_without_noises['cluster_hdbscan'].unique()))
df_hdbscan_clusters_level_second.to_csv('data/cluster_noise_hdbscan_sans_bruit.csv')

# df_hdbscan_clusters_level_second['cluster'].unique()
df_hdbcan_cluster_step_2_without_noises = df_hdbscan_clusters_level_second.\
    merge(
        right=data_dup_images_1,
        on="unique_dup_img_name",
        how="left"
    )
df_hdbcan_cluster_step_2_without_noises.rename(columns={'cluster': 'cluster_hdbscan'}, inplace=True)
df_hdbcan_cluster_step_2_without_noises.to_csv('data/df_hdbcan_cluster_step_2_without_noises.csv') # a considerer

In [ ]:
df_hdbcan_cluster_all_step_noises

In [ ]:
df_hdbcan_cluster_all_step_noises = df_hdbscan_clusters_noise[df_hdbscan_clusters_noise['cluster']==-1]
df_hdbcan_cluster_all_step_noises = df_hdbcan_cluster_all_step_noises.\
    merge(
        right=data_dup_images_1,
        on="unique_dup_img_name",
        how="left"
    )
df_hdbcan_cluster_all_step_noises.rename(columns={'cluster': 'cluster_hdbscan'}, inplace=True)
final_df_hdbscan_clusters = pd.concat([
    df_hdbcan_cluster_step_1_without_noises,
    df_hdbcan_cluster_step_2_without_noises,
    df_hdbcan_cluster_all_step_noises
], ignore_index=True)

# verifions si on a toujours le meme nombre d'images :
assert len(final_df_hdbscan_clusters['image_id'].unique()) == len(data_dup_images_1['image_id'].unique())
assert len(final_df_hdbscan_clusters['unique_dup_img_name'].unique()) == len(data_dup_images_1['unique_dup_img_name'].unique())
print("Verification OKAY")
final_df_hdbscan_clusters.to_csv('data/final_df_hdbscan_clustering.csv')

## visualisation des clusters

In [ ]:
img_cluster_0 = final_df_hdbscan_clusters['unique_dup_img_name'][final_df_hdbscan_clusters['cluster_hdbscan']==0].tolist()
random.seed(42)
random.shuffle(img_cluster_0)
img_cluster_0_to_display = img_cluster_0[:50]
img_analyzer.display_specific_imgs(
    from_internet=False,
    source_path=source_path,
    images_dir = ['img', 'source_img'],
    image_names=img_cluster_0_to_display,
    ncols=7
)

In [ ]:
img_cluster_1 = final_df_hdbscan_clusters['unique_dup_img_name'][final_df_hdbscan_clusters['cluster_hdbscan']==1].tolist()
random.seed(42)
random.shuffle(img_cluster_1)
img_cluster_1_to_display = img_cluster_1
img_analyzer.display_specific_imgs(
    from_internet=False,
    source_path=source_path,
    images_dir = ['img', 'source_img'],
    image_names=img_cluster_1_to_display,
    ncols=10
)

In [ ]:
img_cluster_2 = final_df_hdbscan_clusters['unique_dup_img_name'][final_df_hdbscan_clusters['cluster_hdbscan']==2].tolist()
random.seed(42)
random.shuffle(img_cluster_2)
img_cluster_2_to_display = img_cluster_2
img_analyzer.display_specific_imgs(
    from_internet=False,
    source_path=source_path,
    images_dir = ['img', 'source_img'],
    image_names=img_cluster_2_to_display,
    ncols=7
)

In [ ]:
img_cluster_3 = final_df_hdbscan_clusters['unique_dup_img_name'][
    final_df_hdbscan_clusters['cluster_hdbscan']==3
].tolist()
random.seed(42)
random.shuffle(img_cluster_3)
img_cluster_3_to_display = img_cluster_3
img_analyzer.display_specific_imgs(
    from_internet=False,
    source_path=source_path,
    images_dir=['img', 'source_img'],
    image_names=img_cluster_3_to_display,
    ncols=7
)


In [ ]:
# --- Cluster 4 ---
img_cluster_4 = final_df_hdbscan_clusters['unique_dup_img_name'][
    final_df_hdbscan_clusters['cluster_hdbscan']==4
].tolist()
random.seed(42)
random.shuffle(img_cluster_4)
img_cluster_4_to_display = img_cluster_4[:80]
img_analyzer.display_specific_imgs(
    from_internet=False,
    source_path=source_path,
    images_dir=['img', 'source_img'],
    image_names=img_cluster_4_to_display,
    ncols=7
)


In [ ]:
# --- Cluster 6 ---
img_cluster_6 = final_df_hdbscan_clusters['unique_dup_img_name'][
    final_df_hdbscan_clusters['cluster_hdbscan']==6
].tolist()
random.seed(42)
random.shuffle(img_cluster_6)
img_cluster_6_to_display = img_cluster_6[:50]
img_analyzer.display_specific_imgs(
    from_internet=False,
    source_path=source_path,
    images_dir=['img', 'source_img'],
    image_names=img_cluster_6_to_display,
    ncols=7
)


In [ ]:
# --- Cluster 6 ---
img_cluster_6 = final_df_hdbscan_clusters['unique_dup_img_name'][
    final_df_hdbscan_clusters['cluster_hdbscan']==6
].tolist()
random.seed(42)
random.shuffle(img_cluster_6)
img_cluster_6_to_display = img_cluster_6[:50]
img_analyzer.display_specific_imgs(
    from_internet=False,
    source_path=source_path,
    images_dir=['img', 'source_img'],
    image_names=img_cluster_6_to_display,
    ncols=7
)
